# Notebook 2 — Modelo Mental de OpenMP com Python

Este notebook continua a aula integrada de:

- balanceamento de carga
- consistência de memória
- OpenMP como modelo mental
- MPI/SPMD como execução distribuída

## Objetivo
Entender, de forma prática, o que o OpenMP faz conceitualmente:

- criar uma **região paralela**
- dividir trabalho entre threads
- sincronizar ao final
- lidar com variáveis **shared** e **private**
- perceber quando o overhead compensa ou não

> Aqui não vamos programar OpenMP real em C/C++, mas sim reproduzir sua lógica usando Python.


## Como interpretar este notebook

Pense nos exemplos abaixo como equivalentes conceituais de diretivas clássicas do OpenMP, como:

- `parallel`
- `for`
- `schedule(static)` e `schedule(dynamic)`
- variáveis `shared`
- variáveis `private`
- ponto implícito de sincronização no final da região paralela

A ideia não é copiar sintaxe de OpenMP, mas entender **o comportamento que ele expressa**.


In [ ]:
import math
import os
import time
import random
from concurrent.futures import ThreadPoolExecutor, as_completed

import matplotlib.pyplot as plt


## 1. Região paralela: uma primeira intuição

Em OpenMP, uma região paralela cria múltiplas threads para executar trabalho.  
Em Python, podemos usar `ThreadPoolExecutor` para visualizar essa ideia.

⚠️ Observação importante:
para tarefas realmente CPU-bound, Python com threads sofre com o **GIL**.  
Mesmo assim, o notebook é útil para entender o **modelo mental**.


In [ ]:
def tarefa_simples(thread_id):
    inicio = time.perf_counter()
    time.sleep(0.2)  # I/O artificial para facilitar observar paralelismo com threads
    fim = time.perf_counter()
    return {
        "thread_id": thread_id,
        "tempo": fim - inicio
    }

inicio = time.perf_counter()
with ThreadPoolExecutor(max_workers=4) as ex:
    resultados = list(ex.map(tarefa_simples, range(4)))
fim = time.perf_counter()

print("Resultados:", resultados)
print(f"Tempo total: {fim - inicio:.4f} s")


### Leitura conceitual
Nesse exemplo, todas as threads entram na mesma região paralela, executam uma tarefa e depois retornam.

Em OpenMP, o final da região costuma implicar uma **barreira implícita**:  
o programa só segue quando todas as threads terminam.


## 2. Parallel for: dividir iterações entre threads

Agora vamos simular o equivalente conceitual de um `parallel for`.

Cada iteração executa a mesma lógica, mas sobre dados diferentes.
Isso é exatamente o padrão que aparece em muitos laços paralelizáveis.


In [ ]:
dados = list(range(1, 21))
dados[:5], len(dados)


In [ ]:
def processar_item(x):
    # pequena tarefa simulada
    time.sleep(0.05)
    return x, x * x

inicio = time.perf_counter()
with ThreadPoolExecutor(max_workers=4) as ex:
    saida = list(ex.map(processar_item, dados))
fim = time.perf_counter()

print("Primeiros resultados:", saida[:5])
print(f"Tempo total: {fim - inicio:.4f} s")


### Equivalente mental em OpenMP
Algo como:

```c
#pragma omp parallel for
for (int i = 0; i < N; i++) {
    ...
}
```

A ideia central é:  
**as iterações do laço são distribuídas entre as threads**.


## 3. Schedule estático vs dinâmico

Uma das ideias mais importantes de OpenMP é que nem toda distribuição de iterações é igual.

Se cada iteração tiver custo parecido, um escalonamento estático costuma funcionar bem.  
Se o custo variar muito, o escalonamento dinâmico pode reduzir ociosidade.


In [ ]:
def gerar_cargas():
    # várias tarefas leves e algumas bem mais caras
    return [1,1,1,1,8,1,1,1,10,1,1,7,1,1,1,9,1,1,1,6]

cargas = gerar_cargas()
cargas


In [ ]:
plt.figure(figsize=(10, 3))
plt.bar(range(len(cargas)), cargas)
plt.title("Custo relativo por iteração")
plt.xlabel("Iteração")
plt.ylabel("Carga")
plt.show()


### 3.1 Escalonamento estático

No modo estático, as iterações são pré-divididas entre as threads.
É simples e barato, mas pode gerar desequilíbrio.


In [ ]:
def particionar_estatico(cargas, n_threads):
    partes = [[] for _ in range(n_threads)]
    n = len(cargas)
    chunk = math.ceil(n / n_threads)
    for i in range(n_threads):
        ini = i * chunk
        fim = min((i + 1) * chunk, n)
        partes[i] = list(enumerate(cargas))[ini:fim]
    return partes

def worker_estatico(thread_id, pares):
    inicio = time.perf_counter()
    processadas = []
    for idx, carga in pares:
        time.sleep(carga * 0.03)
        processadas.append((idx, carga))
    fim = time.perf_counter()
    return {
        "thread_id": thread_id,
        "iteracoes": processadas,
        "carga_total": sum(c for _, c in pares),
        "tempo": fim - inicio
    }


In [ ]:
import math

partes_estaticas = particionar_estatico(cargas, 4)
partes_estaticas


In [ ]:
inicio = time.perf_counter()
with ThreadPoolExecutor(max_workers=4) as ex:
    futures = [ex.submit(worker_estatico, tid, pares) for tid, pares in enumerate(partes_estaticas)]
    saidas_estaticas = [f.result() for f in futures]
fim = time.perf_counter()

tempo_total_estatico = fim - inicio
saidas_estaticas, tempo_total_estatico


### 3.2 Escalonamento dinâmico

No modo dinâmico, as threads vão pegando novas iterações à medida que terminam as anteriores.

Esse modelo se aproxima do uso de filas de tarefas.


In [ ]:
import queue
import threading

def worker_dinamico(thread_id, fila, resultados):
    inicio = time.perf_counter()
    carga_total = 0
    iteracoes = []

    while True:
        try:
            idx, carga = fila.get_nowait()
        except queue.Empty:
            break

        time.sleep(carga * 0.03)
        carga_total += carga
        iteracoes.append((idx, carga))
        fila.task_done()

    fim = time.perf_counter()
    resultados.append({
        "thread_id": thread_id,
        "iteracoes": iteracoes,
        "carga_total": carga_total,
        "tempo": fim - inicio
    })


In [ ]:
fila = queue.Queue()
for item in list(enumerate(cargas)):
    fila.put(item)

resultados_dinamicos = []
threads = []

inicio = time.perf_counter()
for tid in range(4):
    t = threading.Thread(target=worker_dinamico, args=(tid, fila, resultados_dinamicos))
    t.start()
    threads.append(t)

for t in threads:
    t.join()
fim = time.perf_counter()

tempo_total_dinamico = fim - inicio
resultados_dinamicos = sorted(resultados_dinamicos, key=lambda x: x["thread_id"])
resultados_dinamicos, tempo_total_dinamico


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].bar(
    [r["thread_id"] for r in saidas_estaticas],
    [r["carga_total"] for r in saidas_estaticas]
)
axes[0].set_title("Carga por thread — estático")
axes[0].set_xlabel("Thread")
axes[0].set_ylabel("Carga total")

axes[1].bar(
    [r["thread_id"] for r in resultados_dinamicos],
    [r["carga_total"] for r in resultados_dinamicos]
)
axes[1].set_title("Carga por thread — dinâmico")
axes[1].set_xlabel("Thread")
axes[1].set_ylabel("Carga total")

plt.tight_layout()
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].bar(
    [r["thread_id"] for r in saidas_estaticas],
    [r["tempo"] for r in saidas_estaticas]
)
axes[0].set_title("Tempo por thread — estático")
axes[0].set_xlabel("Thread")
axes[0].set_ylabel("Tempo (s)")

axes[1].bar(
    [r["thread_id"] for r in resultados_dinamicos],
    [r["tempo"] for r in resultados_dinamicos]
)
axes[1].set_title("Tempo por thread — dinâmico")
axes[1].set_xlabel("Thread")
axes[1].set_ylabel("Tempo (s)")

plt.tight_layout()
plt.show()


In [ ]:
print(f"Tempo total estático: {tempo_total_estatico:.4f} s")
print(f"Tempo total dinâmico: {tempo_total_dinamico:.4f} s")


## 4. Shared vs Private

Esse é um ponto central do OpenMP.

- **shared**: todas as threads enxergam a mesma variável
- **private**: cada thread tem sua própria cópia local

Quando isso é mal compreendido, surgem bugs e resultados incorretos.


### 4.1 Variável shared: cuidado com concorrência

Abaixo, várias threads atualizam uma estrutura compartilhada.  
Isso funciona porque a atualização é controlada com lock.


In [ ]:
contador_global = 0
lock = threading.Lock()

def incrementar_shared(n):
    global contador_global
    for _ in range(n):
        with lock:
            contador_global += 1

contador_global = 0
threads = []

for _ in range(4):
    t = threading.Thread(target=incrementar_shared, args=(50000,))
    t.start()
    threads.append(t)

for t in threads:
    t.join()

print("Valor final do contador_global:", contador_global)


### Leitura conceitual
Quando a variável é compartilhada, o acesso concorrente precisa ser disciplinado.

Em OpenMP, isso lembraria o uso de:

- regiões críticas (`critical`)
- reduções (`reduction`)
- barreiras quando necessário


### 4.2 Variável private: cada thread com seu estado local

Agora cada thread acumula um resultado local, sem interferir diretamente nas demais.


In [ ]:
def soma_local(valores):
    acumulado = 0  # variável local / private
    for v in valores:
        acumulado += v
    return acumulado

partes = [
    [1, 2, 3],
    [4, 5, 6],
    [7, 8],
    [9, 10]
]

with ThreadPoolExecutor(max_workers=4) as ex:
    somas_locais = list(ex.map(soma_local, partes))

print("Somas locais:", somas_locais)
print("Soma final:", sum(somas_locais))


### Leitura conceitual
Esse padrão é muito importante:

1. cada thread trabalha com variável privada;
2. no final, os resultados são combinados.

Esse é justamente o espírito de uma **reduction**.


## 5. Reduction: forma segura e eficiente de combinar resultados

Vamos comparar a ideia de acumulação local seguida de combinação final.


In [ ]:
valores = list(range(1, 1001))

def soma_parcial_bloco(bloco):
    total = 0
    for x in bloco:
        total += x
    return total

def dividir_blocos(lista, n):
    tamanho = math.ceil(len(lista) / n)
    return [lista[i:i+tamanho] for i in range(0, len(lista), tamanho)]

blocos = dividir_blocos(valores, 4)

with ThreadPoolExecutor(max_workers=4) as ex:
    parciais = list(ex.map(soma_parcial_bloco, blocos))

resultado = sum(parciais)

print("Parciais:", parciais)
print("Resultado paralelo:", resultado)
print("Resultado sequencial:", sum(valores))


## 6. Overhead: quando paralelizar não compensa

Nem todo laço deve ser paralelizado.

Se o trabalho por iteração for muito pequeno, o custo de:

- criar threads
- coordenar execução
- sincronizar resultados

pode superar o ganho.


In [ ]:
def trabalho_muito_pequeno(x):
    return x + 1

dados_pequenos = list(range(1000))

inicio = time.perf_counter()
seq = [trabalho_muito_pequeno(x) for x in dados_pequenos]
fim = time.perf_counter()
tempo_seq = fim - inicio

inicio = time.perf_counter()
with ThreadPoolExecutor(max_workers=4) as ex:
    par = list(ex.map(trabalho_muito_pequeno, dados_pequenos))
fim = time.perf_counter()
tempo_par = fim - inicio

print(f"Sequencial: {tempo_seq:.6f} s")
print(f"Paralelo com threads: {tempo_par:.6f} s")
print("Resultados iguais?", seq == par)


### Interpretação
Esse é um dos erros mais comuns em computação paralela:

> tentar paralelizar trechos cujo custo é pequeno demais para pagar o overhead.

Em OpenMP, isso aparece como escolha ruim de granularidade.


## 7. OpenMP mindset: o que levar deste notebook

Até aqui, o que importa não é a sintaxe do OpenMP, mas o raciocínio:

- existe uma região paralela;
- as iterações precisam ser distribuídas;
- o escalonamento afeta o desempenho;
- variáveis shared exigem cuidado;
- variáveis private simplificam o paralelismo;
- reduções são padrão natural para combinar resultados;
- granularidade importa.


## 8. Relação com a aula anterior e com a próxima

### Relação com o Notebook 1
No notebook anterior, vimos que o desempenho depende de **balanceamento de carga**.

### Relação com este notebook
Agora vimos isso no contexto de **memória compartilhada**, no estilo OpenMP.

### Ponte para o Notebook 3
O próximo passo natural é sair do paralelismo local e ir para:

- múltiplos processos
- múltiplos workers independentes
- particionamento explícito de dados
- modelo mental de MPI / SPMD


## 9. Exercícios propostos

### Exercício 1
Modifique a lista `cargas` para criar um caso ainda mais desbalanceado.  
Compare novamente o estático e o dinâmico.

### Exercício 2
Explique por que o modo dinâmico pode reduzir ociosidade, mas também pode introduzir mais overhead.

### Exercício 3
Implemente uma versão com `chunk size`, em que cada thread pega pequenos lotes por vez, em vez de uma iteração por vez.

### Exercício 4
Escreva com suas palavras:
- o que seria `shared`
- o que seria `private`
- quando usar uma abordagem de `reduction`


## 10. Síntese final

Este notebook mostrou o núcleo conceitual do OpenMP:

- paralelismo local em CPU;
- divisão de iterações entre threads;
- sincronização ao final;
- impacto do escalonamento;
- importância de distinguir estado compartilhado de estado privado.

> OpenMP não é “mágica para acelerar código”.
> Ele funciona bem quando o programador entende decomposição, granularidade e custo de sincronização.
